# Exact inference: variable elimination

_Artificial Intelligence — more_

**Sum out the variables you don't care about, one at a time, multiplying factors as you go.**

To answer a query in a Bayes net exactly, you must sum the joint probability over every variable you did not observe and do not care about.

     Done naively, that sum is astronomically large. Variable elimination makes it tractable by pushing sums inward.

---

This notebook is a practice scaffold for the **Exact inference: variable elimination** lesson. Run the cells, then tackle the practice problems at the bottom. _Save a copy to your Drive (File → Save a copy in Drive) to edit and keep your work._

In [ ]:
# Setup — numpy / pandas / scikit-learn / matplotlib ship with Colab.
import numpy as np
import matplotlib.pyplot as plt

## Reference implementation — Python + NumPy

### Step 1 — Write down the two factorsVariable elimination works on *factors* — tables of non-negative numbers over a few variables. Here we have two binary factors: `f1(A, B)` and `f2(B, C)`, each a 2x2 table indexed by its two variables. The shared variable `B` is the one we are about to sum out.

In [ ]:
import numpy as np

# binary factors as 2x2 tables, indexed [A,B] and [B,C].
f1 = np.array([[2.0, 1.0],    # f1(A=0,B=0)=2, f1(A=0,B=1)=1
               [1.0, 2.0]])   #              A=1 row
f2 = np.array([[1.0, 2.0],    # f2(B=0,C=0)=1, f2(B=0,C=1)=2
               [3.0, 1.0]])   #              B=1 row
print("f1 =\n", f1)
print("f2 =\n", f2)

### Step 2 — Sum out BEliminating `B` means: for every combination of `A` and `C`, multiply the two factors over each value of `B` and add the results — `g(A, C) = sum_B f1[A, B] * f2[B, C]`. That is exactly a matrix product over the shared `B` axis, which `np.einsum` expresses directly. We spot-check one entry: `g(A=0, C=0) = 2*1 + 1*3 = 5`.

In [ ]:
# eliminate B: g(A,C) = sum_B f1[A,B] * f2[B,C]  (a matrix product over B)
g = np.einsum("ab,bc->ac", f1, f2)
print("g(A,C) factor after summing out B:")
print(g)
print("g(A=0,C=0) =", g[0, 0], "(expect 2*1 + 1*3 = 5)")

### Step 3 — Normalize to a distributionThe factor `g(A, C)` still holds un-normalized weights. Dividing by their total turns it into a proper probability distribution over the remaining variables `(A, C)` — the answer to the query, with the eliminated variable `B` already accounted for.

In [ ]:
# normalize to a distribution over (A,C) for the query answer
total = g.sum()
distribution = g / total
print("normalized:")
print(np.round(distribution, 3))

## Visualize the data & results

_Sprinkler-Rain-WetGrass network: the grass is wet — what is the posterior probability that it rained?_

### Step 1 — Build the Sprinkler–Rain–WetGrass networkThis is the classic Bayes net (Russell & Norvig): a hidden `Cloudy` cause influences both `Sprinkler` and `Rain`, and both of those determine whether the grass is wet. We encode each conditional probability table as a NumPy array, with `0 = False` and `1 = True`. `P_W[s, r]` gives `P(WetGrass | Sprinkler=s, Rain=r)`.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

# Sprinkler-Rain-WetGrass Bayes net (Russell & Norvig). 0=False, 1=True.
P_C = np.array([0.5, 0.5])                    # P(Cloudy)
P_S_C = np.array([[0.5, 0.5], [0.9, 0.1]])    # P(Sprinkler | Cloudy)
P_R_C = np.array([[0.8, 0.2], [0.2, 0.8]])    # P(Rain | Cloudy)

P_W = np.zeros((2, 2, 2))                      # P(WetGrass | Sprinkler, Rain)
P_W[0, 0] = [1.0, 0.0]
P_W[0, 1] = [0.1, 0.9]
P_W[1, 0] = [0.1, 0.9]
P_W[1, 1] = [0.01, 0.99]
print("CPTs built. P(WetGrass=True | S=1, R=1) =", P_W[1, 1, 1])

### Step 2 — Eliminate Cloudy and Sprinkler, condition on wet grassWe want `P(Rain | WetGrass = True)`. The query variable is `Rain`, so we sum the joint over the variables we do not care about — `Cloudy` and `Sprinkler` — while fixing the observed evidence `WetGrass = True` (index `1` in the last axis of `P_W`). Normalizing the resulting two-entry factor gives the posterior over rain.

In [ ]:
# Eliminate Cloudy and Sprinkler; keep Rain, condition on WetGrass=True.
joint_R = np.zeros(2)
for c in range(2):
    for s in range(2):
        for r in range(2):
            joint_R[r] += P_C[c] * P_S_C[c, s] * P_R_C[c, r] * P_W[s, r, 1]

post = joint_R / joint_R.sum()                 # posterior over Rain, ~[0.292, 0.708]
print("P(Rain | WetGrass=True) =", np.round(post, 3))

### Step 3 — Plot the posterior over RainThe bar chart shows the two posterior probabilities. Seeing wet grass makes rain the more likely explanation (about `0.708`) than no rain (about `0.292`) — variable elimination has turned the network plus one observation into a clean answer.

In [ ]:
labels = ["Rain = False", "Rain = True"]
colors = ["#ff7b72", "#4ea1ff"]

fig, ax = plt.subplots()
bars = ax.bar(labels, post, color=colors)
for b, v in zip(bars, post):
    ax.text(b.get_x() + b.get_width()/2, v, str(round(v, 3)), ha="center", va="bottom")
ax.set_title("P(Rain | WetGrass = True) by variable elimination")
plt.show()